In [1]:
import pandas as pd

labels = pd.read_csv("../data/labels/labeled.csv", encoding="utf-8-sig")
key = pd.read_csv("../data/labels/sample_key.csv", dtype={"CAMPNO": str})

ev = labels.merge(key, on="sample_id", validate="one_to_one")
print(ev.shape)
print(ev.groupby("stratum")["label"].agg(["count", "sum", "mean"]))

(200, 11)
         count  sum  mean
stratum                  
A          100   81  0.81
B           25    1  0.04
C           25    0  0.00
D           50    0  0.00


In [2]:
a = ev[ev["stratum"] == "A"]
print("Precision (all A):", round(a["label"].mean(), 3))

main = a[a["battery_type"] != "Battery (text only)"]
print("Precision (main set, component-based):", round(main["label"].mean(), 3), "n =", len(main))

Precision (all A): 0.81
Precision (main set, component-based): 0.931 n = 72


In [3]:
N = {"A": 347, "B": 235, "C": 2620, "D": 10519}

rates = ev.groupby("stratum")["label"].mean()
est_pos = {s: rates[s] * N[s] for s in N}
print({s: round(v, 1) for s, v in est_pos.items()})

recall = est_pos["A"] / sum(est_pos.values())
print("Estimated recall:", round(recall, 3))

{'A': np.float64(281.1), 'B': np.float64(9.4), 'C': np.float64(0.0), 'D': np.float64(0.0)}
Estimated recall: 0.968


In [4]:
from math import sqrt

def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return round(centre - half, 3), round(centre + half, 3)

print("All A:", wilson(int(a["label"].sum()), len(a)))
print("Main set:", wilson(int(main["label"].sum()), len(main)))

All A: (0.722, 0.875)
Main set: (0.848, 0.97)
